# Representation-to-Image Decoder Training

This notebook loads a pretrained checkpoint (e.g., `resnet18`, `resnet18_mixer`, `split_resnet`, or `split_resnet_mixer`), freezes the encoder, and trains a decoder to reconstruct images from the encoder representation.

> **Important:** The decoder is trained on the **full dataset** (training + testing splits combined), not only the training split.

In [1]:
from __future__ import annotations

import copy
import math
import random
import sys
from pathlib import Path

import os
os.chdir('/home/fidelrio/archive/scalable-compositional-generalization/')


import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import ConcatDataset, DataLoader, Subset
from omegaconf import OmegaConf

from visgen.datasets import Cars3D, CLEVR, DSprites, IRAVEN, MPI3D, Shapes3D
from visgen.models import get_model

plt.rcParams["figure.figsize"] = (10, 4)

In [2]:
# ============ User configuration ============
# Preferred: point to the experiment cfg saved during training.
RUN_CFG_PATH = None  # e.g., "outputs/.../cfg.yml"

# Fallback if RUN_CFG_PATH is None:
DATASET = 'mpi3d'
# DATASET = 'shapes3d'
# MODEL = 'split_resnet_mixer_red_256'
# CHECKPOINT_PATH = f"notebooks/checkpoints/{DATASET}_ain_licg_256_best.pth"
MODEL = 'split_resnet_algebraic_non_iid'
CHECKPOINT_PATH = f"notebooks/checkpoints/{DATASET}_ain_alg_best.pth.tar"

BASE_CFG_PATH = "configs/base.yml"
DATA_CFG_PATH = f"configs/datasets/{DATASET}_non_iid.yml"
MODEL_CFG_PATH = f"configs/models/{MODEL}.yml"
EXPERIMENT_CFG_PATH = "configs/experiments/iid.yml"

# Checkpoint from this repository's training code
# CHECKPOINT_PATH = f"notebooks/checkpoints/{DATASET}_{MODEL}_best.pth"

# Decoder training hyperparameters
SEED = 7
BATCH_SIZE = 64
NUM_WORKERS = 6
EPOCHS = 20
LR = 1e-4
WEIGHT_DECAY = 1e-6
SUBSET = 0.1

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Using device: cuda


In [3]:
def set_seed(seed: int = 0):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def load_cfg(
    run_cfg_path: str | None,
    base_cfg_path: str,
    data_cfg_path: str,
    model_cfg_path: str,
    experiment_cfg_path: str,
):
    if run_cfg_path is not None:
        cfg = OmegaConf.load(run_cfg_path)
        print(f"Loaded run config: {run_cfg_path}")
        return cfg

    cfg = OmegaConf.merge(
        OmegaConf.load(base_cfg_path),
        OmegaConf.load(model_cfg_path),
        OmegaConf.load(data_cfg_path),
        OmegaConf.load(experiment_cfg_path),
    )
    cfg["device"] = DEVICE
    print("Loaded merged cfg from base/data/model/experiment YAML files.")
    return cfg


def _to_dict(cfg_node):
    return OmegaConf.to_container(cfg_node, resolve=True)


def build_full_dataset(data_cfg, subset=1.0):
    dataset_map = {
        "dsprites": DSprites,
        "iraven": IRAVEN,
        "mpi3d": MPI3D,
        "shapes3d": Shapes3D,
        "cars3d": Cars3D,
        "clevr": CLEVR,
    }

    tr_cfg = _to_dict(data_cfg.training)
    te_cfg = _to_dict(data_cfg.testing)

    dataset_name = tr_cfg["dataset"]
    dataset_cls = dataset_map[dataset_name]

    train_dataset = dataset_cls(**tr_cfg)
    test_dataset = dataset_cls(**te_cfg)
    print('len(train_dataset): ', len(train_dataset))
    print('len(test_dataset): ', len(test_dataset))

    full_dataset = ConcatDataset([train_dataset, test_dataset])

    if subset < 1:
        # Get total length
        total_size = len(full_dataset)
        subset_size = int(subset * total_size)  # Take 20% of the data
        
        # Generate random indices and select a subset
        indices = torch.randperm(total_size).tolist()
        subset_dataset = Subset(full_dataset, indices[:subset_size])
        full_dataset = subset_dataset
    
    print(f"Dataset: {dataset_name}")
    print(f"  train split size: {len(train_dataset)}")
    print(f"  test split size:  {len(test_dataset)}")
    print(f"  full size used for decoder training: {len(full_dataset)}")

    return full_dataset


def load_checkpoint_flexible(model: nn.Module, checkpoint_path: str, device: str):
    ckpt = torch.load(checkpoint_path, map_location=device)

    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        state_dict = ckpt["model_state_dict"]
        print("Loaded `model_state_dict` from trainer checkpoint format.")
    elif isinstance(ckpt, dict):
        state_dict = ckpt
        print("Loaded checkpoint as a raw state_dict dictionary.")
    else:
        raise TypeError("Unsupported checkpoint format.")

    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    print(f"Missing keys: {len(missing)}")
    print(f"Unexpected keys: {len(unexpected)}")
    if len(missing) > 0:
        print("First missing keys:", missing[:10])
    if len(unexpected) > 0:
        print("First unexpected keys:", unexpected[:10])

    return model


def ensure_nchw(x: torch.Tensor) -> torch.Tensor:
    # Some models/trainers may hand over [B, V, C, H, W]; use the last view.
    if x.dim() == 5:
        x = x[:, -1]
    return x

In [4]:
class RepresentationDecoder(nn.Module):
    """Simple MLP decoder from representation vector -> image tensor."""

    def __init__(self, rep_dim: int, image_shape: tuple[int, int, int], hidden_dim: int = 2048):
        super().__init__()
        c, h, w = image_shape
        out_dim = c * h * w
        self.image_shape = image_shape
        # self.net = nn.Sequential(
        #     nn.Linear(rep_dim, hidden_dim),
        #     nn.ReLU(inplace=True),
        #     nn.Linear(hidden_dim, hidden_dim),
        #     nn.ReLU(inplace=True),
        #     nn.Linear(hidden_dim, out_dim),
        #     nn.Sigmoid(),
        # )
        self.net = nn.Sequential(
            nn.Linear(in_features=rep_dim, out_features=256),
            nn.ReLU(inplace=True),
            nn.Linear(in_features=256, out_features=1024),
            nn.ReLU(inplace=True),
            nn.Unflatten(dim=1, unflattened_size=[64, 4, 4]),
            nn.ConvTranspose2d(in_channels=64, out_channels=64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(in_channels=64, out_channels=32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(in_channels=32, out_channels=32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(in_channels=32, out_channels=c, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid(),
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        x = self.net(z)
        c, h, w = self.image_shape
        return x.view(z.shape[0], c, h, w)

In [5]:
# def set_training_targets(cfg, dataset):
#     split_attributes_by_dataset = {
#         "dsprites": "scale_shape_x-position_y-position",
#         "iraven": "size_type_color",
#         "cars3d": "elevation_type_orientation",
#         "shapes3d": "wall_floor_object_scale_shape",
#         "clevr": "shape_size_material_color",
#         "mpi3d": "color_shape_height_bgcolor_x-axis_y-axis",
#     }

#     if dataset not in split_attributes_by_dataset:
#         raise ValueError(f"Unknown dataset: {dataset}")

#     # cfg.setdefault("data", {})
#     # cfg["data"].setdefault("training", {})
#     cfg["data"]["training"]["targets"] = split_attributes_by_dataset[dataset]
#     return cfg

def set_training_targets(cfg, dataset):
    dataset_config = {
        "dsprites": {
            "C": [1],
            "D": [2, 3, 14, 14],
            "split_attributes": "scale_shape_x-position_y-position",
        },
        "iraven": {
            "C": [1],
            "D": [6, 3, 3],
            "split_attributes": "size_type_color",
        },
        "cars3d": {
            "C": [1],
            "D": [15, 2, 113],
            "split_attributes": "elevation_type_orientation",
        },
        "shapes3d": {
            "C": [1],
            "D": [7, 7, 7, 6, 3],
            "split_attributes": "wall_floor_object_scale_shape",
        },
        "clevr": {
            "C": [1],
            "D": [2, 2, 1, 7],
            "split_attributes": "shape_size_material_color",
        },
        "mpi3d": {
            "C": [1],
            "D": [5, 4, 2, 2, 34, 34],
            "split_attributes": "color_shape_height_bgcolor_x-axis_y-axis",
        },
    }

    if dataset not in dataset_config:
        raise ValueError(f"Unknown dataset: {dataset}")

    # Ensure structure exists (safer than assuming it)
    cfg.setdefault("data", {})
    cfg["data"].setdefault("training", {})

    cfg["data"]["training"]["targets"] = dataset_config[dataset]["split_attributes"]
    cfg["data"]["training"]["c"] = dataset_config[dataset]["C"][0]
    cfg["data"]["training"]["attr_difficulty"] = dataset_config[dataset]["D"]
    cfg["data"]["testing"]["c"] = dataset_config[dataset]["C"][0]
    cfg["data"]["testing"]["attr_difficulty"] = dataset_config[dataset]["D"]

    cfg["data"]["training"]["path"] = '/workspace1/asoto/fidelrio/paper-alain/' + cfg["data"]["training"]["path"]
    # cfg["data"]["testing"]["path"] = '/workspace1/asoto/fidelrio/paper-alain/' + cfg["data"]["testing"]["path"]

    return cfg

def set_split(cfg):
    if cfg["data"]["training"]["split"] not in ('composition', 'general_composition'):
        cfg["data"]["training"]["split"] = 'general_composition' 
        cfg["data"]["testing"]["split"] = 'general_composition'
    return cfg

In [6]:
set_seed(SEED)

cfg = load_cfg(
    RUN_CFG_PATH,
    BASE_CFG_PATH,
    DATA_CFG_PATH,
    MODEL_CFG_PATH,
    EXPERIMENT_CFG_PATH,
)

cfg = set_training_targets(cfg, DATASET)
cfg = set_split(cfg)

model = get_model(cfg).to(DEVICE)
model = load_checkpoint_flexible(model, CHECKPOINT_PATH, DEVICE)
model.eval()

for p in model.parameters():
    p.requires_grad = False

full_dataset = build_full_dataset(cfg.data, subset=SUBSET)

loader = DataLoader(
    full_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE == "cuda"),
)

# infer representation and image dimensions
sample_x, _ = full_dataset[0]
sample_x = ensure_nchw(sample_x.unsqueeze(0)).to(DEVICE)
with torch.no_grad():
    sample_z = model.extract_representation(sample_x)

rep_dim = int(sample_z.shape[-1])
image_shape = tuple(sample_x.shape[1:])

print(f"Representation dim: {rep_dim}")
print(f"Image shape: {image_shape}")

decoder = RepresentationDecoder(rep_dim=rep_dim, image_shape=image_shape).to(DEVICE)
optimizer = torch.optim.Adam(decoder.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
print(decoder)

Loaded merged cfg from base/data/model/experiment YAML files.
Loaded `model_state_dict` from trainer checkpoint format.
Missing keys: 0
Unexpected keys: 0
[5, 4, 2, 2, 34, 34]
0.5446296296296296
[5, 4, 2, 2, 34, 34]
0.5446296296296296
len(train_dataset):  564672
len(test_dataset):  472128
Dataset: mpi3d
  train split size: 564672
  test split size:  472128
  full size used for decoder training: 103680


/home/fidelrio/.pyenv/versions/3.10.15/envs/systematicity-v2/lib/python3.10/site-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Representation dim: 3072
Image shape: (3, 64, 64)
RepresentationDecoder(
  (net): Sequential(
    (0): Linear(in_features=3072, out_features=256, bias=True)
    (1): ReLU(inplace=True)
    (2): Linear(in_features=256, out_features=1024, bias=True)
    (3): ReLU(inplace=True)
    (4): Unflatten(dim=1, unflattened_size=[64, 4, 4])
    (5): ConvTranspose2d(64, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): ConvTranspose2d(64, 32, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): ConvTranspose2d(32, 32, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (10): ReLU(inplace=True)
    (11): ConvTranspose2d(32, 3, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (12): Sigmoid()
  )
)


In [ ]:
from tqdm.notebook import tqdm

OUTPUT_DECODER_PATH = f"notebooks/outputs/{DATASET}_{MODEL}_decoder.pt"
loaded_results = None
if os.path.exists(OUTPUT_DECODER_PATH):
    loaded_results = torch.load(OUTPUT_DECODER_PATH)
    decoder.load_state_dict(loaded_results["decoder_state_dict"])
    rep_dim = loaded_results["rep_dim"]
    image_shape = loaded_results["image_shape"]
    loss_history = loaded_results["loss_history"]
    
if not loaded_results:
    loss_history = []
    
    for epoch in range(1, EPOCHS + 1):
        decoder.train()
        epoch_loss = 0.0
        num_items = 0
    
        for x, _ in tqdm(loader):
            # x = ensure_nchw(x).to(DEVICE, non_blocking=True).float()
            if x.dim() == 5:
                x = x[:, -1]
            x = x.to(DEVICE, non_blocking=True).float()
             
            with torch.no_grad():
                # z = model.extract_representation(x)
                if x.dim() == 5:
                    x = x[:, -1]
                _, z, _ = model._encode_split(x)
    
            x_hat = decoder(z)
            loss = F.mse_loss(x_hat, x)
    
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
    
            batch_size = x.shape[0]
            epoch_loss += loss.item() * batch_size
            num_items += batch_size
    
        epoch_loss /= max(num_items, 1)
        loss_history.append(epoch_loss)
        print(f"Epoch {epoch:03d}/{EPOCHS:03d} - MSE: {epoch_loss:.6f}")

  0%|          | 0/1620 [00:00<?, ?it/s]

In [ ]:
plt.figure()
plt.plot(loss_history)
plt.title("Decoder training loss (full dataset)")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.grid(True)
plt.show()

In [ ]:
@torch.no_grad()
def show_reconstructions(model, decoder, dataset, n=8):
    model.eval()
    decoder.eval()

    idxs = torch.randperm(len(dataset))[:n].tolist()
    xs = []
    for i in idxs:
        x, _ = dataset[i]
        x = ensure_nchw(x.unsqueeze(0)).squeeze(0)
        xs.append(x)

    x = torch.stack(xs, dim=0).to(DEVICE).float()
    z = model.extract_representation(x)
    x_hat = decoder(z).cpu()
    x = x.cpu()

    c = x.shape[1]
    fig, axes = plt.subplots(2, n, figsize=(2*n, 4))
    for i in range(n):
        xi = x[i].permute(1, 2, 0).numpy() if c > 1 else x[i, 0].numpy()
        xhi = x_hat[i].permute(1, 2, 0).numpy() if c > 1 else x_hat[i, 0].numpy()

        if c > 1:
            axes[0, i].imshow(xi.clip(0, 1))
            axes[1, i].imshow(xhi.clip(0, 1))
        else:
            axes[0, i].imshow(xi, cmap="gray", vmin=0.0, vmax=1.0)
            axes[1, i].imshow(xhi, cmap="gray", vmin=0.0, vmax=1.0)

        axes[0, i].axis("off")
        axes[1, i].axis("off")

    axes[0, 0].set_ylabel("Original")
    axes[1, 0].set_ylabel("Reconstruction")
    plt.tight_layout()
    plt.show()

show_reconstructions(model, decoder, full_dataset, n=8)

In [ ]:
# Optional: save decoder weights
OUTPUT_DECODER_PATH = f"notebooks/outputs/{DATASET}_{MODEL}_decoder.pt"
torch.save(
    {
        "decoder_state_dict": decoder.state_dict(),
        "rep_dim": rep_dim,
        "image_shape": image_shape,
        "epochs": EPOCHS,
        "loss_history": loss_history,
    },
    OUTPUT_DECODER_PATH,
)
print(f"Saved decoder to: {OUTPUT_DECODER_PATH}")

In [ ]:
# def mixer_representation(model, x):
#     x = x.to(DEVICE)
    
#     batch_size, num_views = x.shape[:2]
#     x_flat = x.reshape(batch_size * num_views, *x.shape[2:])
#     reps_flat = model._encode(x_flat)
#     reps = reps_flat.view(batch_size, num_views, -1)
    
#     target_idx = 3
#     input_indices = [0, 1, 2]
#     mixer_inputs = reps[:, input_indices, :]
#     target_rep = reps[:, target_idx, :]
#     mixed_rep = model.mixer_input_projection(mixer_inputs)
#     mixed_rep = model.mixer(mixed_rep)
#     mixed_rep = model.mixer_output_projection(mixed_rep)
#     # return target_rep, mixed_rep.detach()
#     return target_rep, reps[:,0,:].detach()

# def split_mixer_representation(model, x):
#     x = x.to(DEVICE)
    
#     batch_size, num_views = x.shape[:2]
#     x_flat = x.reshape(batch_size * num_views, *x.shape[2:])
#     x_split_flat, reps_flat, _ = model._encode_split(x_flat)
#     reps = reps_flat.view(batch_size, num_views, -1)
    
#     target_idx = 3
#     input_indices = [0, 1, 2]
#     mixer_inputs = reps[:, input_indices, :]
#     target_rep = reps[:, target_idx, :]
#     mixed_rep = model._project_rep_pieces(mixer_inputs)
#     mixed_rep = model.mixer(mixed_rep)
#     mixed_rep = model.mixer_output_projection(mixed_rep)

#     return target_rep, mixed_rep.detach()
#     # return target_rep, reps[:,0,:].detach()

def split_mixer_representation(model, x):
    x = x.to(DEVICE)
    
    batch_size, num_views = x.shape[:2]
    x_flat = x.reshape(batch_size * num_views, *x.shape[2:])
    x_split_flat, reps_flat, _ = model._encode_split(x_flat)
    reps = reps_flat.view(batch_size, num_views, -1)

    rep_ac = reps[:, 0, :]
    rep_ad = reps[:, 1, :]
    rep_bc = reps[:, 2, :]
    rep_bd = reps[:, 3, :]
    residual = rep_ac - rep_ad - rep_bc + rep_bd

    target_rep = rep_bd
    mixed_rep = rep_bc + rep_ad - rep_ac
    return target_rep.detach(), mixed_rep.detach()

def show_mixer_reconstructions(model, decoder, dataset, n=8):
    model.eval()
    decoder.eval()

    idxs = torch.randperm(len(dataset))[:n].tolist()
    xs = []
    for i in idxs:
        x, _ = dataset[i]
        # x = ensure_nchw(x.unsqueeze(0)).squeeze(0)
        xs.append(x)

    x = torch.stack(xs, dim=0).to(DEVICE).float()
    print(x.shape)
    # z = mixer_representation(model, x)
    z_true, z = split_mixer_representation(model, x)
    x_true_hat = decoder(z_true).detach().cpu()
    x_hat = decoder(z).detach().cpu()
    x = x.cpu()

    print('z dist: :', float(((z_true - z) ** 2).mean()))
    print('x dist: :', float(((x_true_hat - x_hat) ** 2).mean()))

    c = x.shape[1]
    fig, axes = plt.subplots(6, n, figsize=(1.75*n, 10.5))
    for i in range(n):
        xi1 = x[i][0].permute(1, 2, 0).numpy() if c > 1 else x[i, 0].numpy()
        xi2 = x[i][1].permute(1, 2, 0).numpy() if c > 1 else x[i, 0].numpy()
        xi3 = x[i][2].permute(1, 2, 0).numpy() if c > 1 else x[i, 0].numpy()
        xi4 = x[i][3].permute(1, 2, 0).numpy() if c > 1 else x[i, 0].numpy()
        xthi = x_true_hat[i].permute(1, 2, 0).numpy() if c > 1 else x_hat[i, 0].numpy()
        xhi = x_hat[i].permute(1, 2, 0).numpy() if c > 1 else x_hat[i, 0].numpy()

        if c > 1:
            axes[0, i].imshow(xi1.clip(0, 1))
            axes[1, i].imshow(xi2.clip(0, 1))
            axes[2, i].imshow(xi3.clip(0, 1))
            axes[3, i].imshow(xi4.clip(0, 1))
            axes[4, i].imshow(xthi.clip(0, 1))
            axes[5, i].imshow(xhi.clip(0, 1))
        else:
            axes[0, i].imshow(xi1, cmap="gray", vmin=0.0, vmax=1.0)
            axes[1, i].imshow(xi2, cmap="gray", vmin=0.0, vmax=1.0)
            axes[2, i].imshow(xi3, cmap="gray", vmin=0.0, vmax=1.0)
            axes[3, i].imshow(xi4, cmap="gray", vmin=0.0, vmax=1.0)
            axes[4, i].imshow(xthi, cmap="gray", vmin=0.0, vmax=1.0)
            axes[5, i].imshow(xhi, cmap="gray", vmin=0.0, vmax=1.0)

        axes[0, i].axis("off")
        axes[1, i].axis("off")
        axes[2, i].axis("off")
        axes[3, i].axis("off")
        axes[4, i].axis("off")
        axes[5, i].axis("off")

    axes[0, 0].set_ylabel("Original")
    axes[1, 0].set_ylabel("Reconstruction")
    plt.tight_layout()
    plt.show()

In [ ]:
from visgen.datasets.non_iid import NonIIDWrapper
trainset, testset = full_dataset.dataset.datasets
non_iid_train_dataset = NonIIDWrapper(trainset, shared_other_attributes=True)
non_iid_test_dataset = NonIIDWrapper(testset, shared_other_attributes=True)

show_mixer_reconstructions(model, decoder, non_iid_test_dataset, n=8)

In [ ]:
@torch.no_grad()
def mixer_distance_samples(model, decoder, dataloader, max_samples=512, device=DEVICE):
    model.eval()
    decoder.eval()

    all_z_dists = []
    all_x_dists = []

    n_seen = 0

    for x, y in dataloader:
        # Handles datasets returning (x, y) or just x
        x = x.to(device).float()
        z_true, z_mix = split_mixer_representation(model, x)

        x_true_hat = decoder(z_true)
        x_mix_hat = decoder(z_mix)

        # Per-sample distances
        z_dist = ((z_true - z_mix) ** 2).mean(dim=1)

        x_dist = ((x_true_hat - x_mix_hat) ** 2).flatten(start_dim=1).mean(dim=1)

        all_z_dists.append(z_dist.detach().cpu())
        all_x_dists.append(x_dist.detach().cpu())

        n_seen += x.shape[0]
        if n_seen >= max_samples:
            break

    z_dists = torch.cat(all_z_dists)[:max_samples]
    x_dists = torch.cat(all_x_dists)[:max_samples]

    return z_dists, x_dists


def plot_latent_vs_image_distances(
    model,
    decoder,
    train_loader,
    test_loader,
    max_samples=512,
    device=DEVICE,
):
    train_z, train_x = mixer_distance_samples(
        model, decoder, train_loader, max_samples=max_samples, device=device
    )

    test_z, test_x = mixer_distance_samples(
        model, decoder, test_loader, max_samples=max_samples, device=device
    )

    plt.figure(figsize=(7, 5))


    plt.scatter(
        test_z.numpy(),
        test_x.numpy(),
        alpha=0.6,
        label="Test",
        s=25,
    )

    plt.scatter(
        train_z.numpy(),
        train_x.numpy(),
        alpha=0.6,
        label="Train",
        s=25,
    )

    # print(train_z[:2])
    # print(train_x[:2])
    # print(test_z[:2])
    # print(test_x[:2])

    plt.xlabel("Latent-space distance: MSE(z_true, z_mix)")
    plt.ylabel("Image-space distance: MSE(dec(z_true), dec(z_mix))")
    plt.title("Latent vs Image Distance for Mixer Reconstructions")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
non_iid_train_loader = DataLoader(
    non_iid_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE == "cuda"),
)
non_iid_test_loader = DataLoader(
    non_iid_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE == "cuda"),
)

plot_latent_vs_image_distances(
    model=model,
    decoder=decoder,
    train_loader=non_iid_train_loader,
    test_loader=non_iid_test_loader,
    max_samples=512,
)